# Gemma 4 Voice Calling Agent

A conversational voice agent powered by Gemma 4.
**You speak** into the mic, the agent **listens, thinks, and speaks back**.

### Prerequisites
Make sure the Gemma 4 server is running:
```bash
cd ~/gemma-server && source venv/bin/activate && python server.py
```

## 0. Install Dependencies

In [4]:
!pip install -q langchain langchain-openai langchain-core openai SpeechRecognition gTTS pydub

'pip' is not recognized as an internal or external command,
operable program or batch file.


## 1. Connect to Gemma 4

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

llm = ChatOpenAI(
    model="gemma-4-e2b-it",
    base_url="http://localhost:8000/v1",
    api_key="not-needed",
    temperature=0.7,
    max_tokens=256,
)

print("Connected to Gemma 4")

Connected to Gemma 4


## 2. Voice Calling Agent

Run this cell to start a voice conversation with Gemma 4.
- Click the **record** button to speak
- The agent will **listen**, **think**, and **respond with voice**
- Conversation history is maintained across turns

In [5]:
import speech_recognition as sr
from gtts import gTTS
from IPython.display import Audio, display, clear_output
import tempfile
import os

# Agent personality
SYSTEM_PROMPT = (
    "You are a helpful voice assistant named Gemma. "
    "Keep your responses concise and conversational (2-3 sentences max) "
    "since the user is talking to you by voice."
)

# Conversation memory
history = [SystemMessage(content=SYSTEM_PROMPT)]
recognizer = sr.Recognizer()

def listen():
    """Record from microphone and return text."""
    with sr.Microphone() as source:
        print("Adjusting for ambient noise...")
        recognizer.adjust_for_ambient_noise(source, duration=1)
        print("Listening... (speak now)")
        audio = recognizer.listen(source, timeout=10, phrase_time_limit=15)
    print("Processing speech...")
    text = recognizer.recognize_google(audio)
    return text

def speak(text):
    """Convert text to speech and play it."""
    tts = gTTS(text=text, lang='en')
    with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as f:
        tts.save(f.name)
        display(Audio(f.name, autoplay=True))

def agent_respond(user_text):
    """Send user text to Gemma and get response."""
    history.append(HumanMessage(content=user_text))
    response = llm.invoke(history)
    history.append(AIMessage(content=response.content))
    return response.content

print("Voice Calling Agent ready!")
print("Run the next cell to start a call.")

Voice Calling Agent ready!
Run the next cell to start a call.


## 3. Start a Call
Run this cell each time you want to speak. The agent will listen and respond.

In [6]:
try:
    # Listen
    user_text = listen()
    print(f"You: {user_text}")
    
    # Think
    print("Gemma is thinking...")
    response = agent_respond(user_text)
    print(f"Gemma: {response}")
    
    # Speak
    speak(response)
    
except sr.WaitTimeoutError:
    print("No speech detected. Run the cell again.")
except sr.UnknownValueError:
    print("Could not understand audio. Try again.")
except Exception as e:
    print(f"Error: {e}")

Error: Could not find PyAudio; check installation


## 4. Use an Audio File Instead
If you don't have a microphone, use a pre-recorded audio file.

In [8]:
# Load audio file instead of microphone
AUDIO_FILE = r"C:\Users\User\voice.wav"

print("Your audio:")
display(Audio(filename=AUDIO_FILE))

# Transcribe
with sr.AudioFile(AUDIO_FILE) as source:
    audio = recognizer.record(source)
    user_text = recognizer.recognize_google(audio)

print(f"\nYou said: {user_text}")

# Get Gemma's response
print("\nGemma is thinking...")
response = agent_respond(user_text)
print(f"Gemma: {response}")

# Speak it back
speak(response)

Your audio:



You said: we are hiring please see your openings at my signature care.com we are also hosting many hiring fears around the area compared in person or meet with a team member Virtually visit my signature care.org flash careers for followers on Instagram

Gemma is thinking...
Gemma: That sounds great! You should definitely check out signaturecare.com for the job openings and keep an eye out for those hiring fairs. Good luck with your recruitment!


## 5. Continuous Conversation Loop
Run this for a multi-turn phone call experience. Say **"goodbye"** to hang up.

In [ ]:
import time

print("Starting call... Say 'goodbye' or 'bye' to hang up.")
speak("Hello! I'm Gemma, your AI assistant. How can I help you today?")

while True:
    try:
        time.sleep(1)  # Brief pause between turns
        user_text = listen()
        print(f"You: {user_text}")
        
        # Check for hang up
        if any(word in user_text.lower() for word in ['goodbye', 'bye', 'hang up', 'end call']):
            print("Gemma: Goodbye! It was nice talking to you.")
            speak("Goodbye! It was nice talking to you.")
            break
        
        response = agent_respond(user_text)
        print(f"Gemma: {response}")
        speak(response)
        
    except sr.WaitTimeoutError:
        print("(silence detected, still listening...)")
        continue
    except sr.UnknownValueError:
        print("(didn't catch that, still listening...)")
        continue
    except KeyboardInterrupt:
        print("\nCall ended.")
        break

print(f"\nCall summary: {len(history) - 1} messages exchanged.")

## 6. View Conversation History
See the full transcript of your call.

In [ ]:
print("=" * 50)
print("CALL TRANSCRIPT")
print("=" * 50)
for msg in history[1:]:  # skip system prompt
    role = "You" if msg.type == "human" else "Gemma"
    print(f"{role}: {msg.content}")
    print("-" * 30)

## 7. Reset Conversation
Clear history to start a fresh call.

In [ ]:
history = [SystemMessage(content=SYSTEM_PROMPT)]
print("Conversation history cleared. Ready for a new call.")